In [31]:
import sqlite3
import pandas as pd
import xgboost as xgb
import numpy as np

upcoming_opponent = "Hoston Celtics"  
month_of_game = 2 

# List everyone who is playing for your team today
active_lineup = [
    "Konstantinos Spanakis",
    "Giorgos Lamprinidis",
    "Michail Alfaras",
    "Thodoris Lamprinidis",
    "Giannis Lamprinidis",
    "Nikolaos Lekakis",
    "Dionusis Miliopoulos"
]


# ==========================================


In [32]:

# 2. LOAD MODEL & FEATURES
model = xgb.XGBRegressor()
model.load_model("../model/xgb_model.json")

# Added the new Alpha Dog feature at the bottom!
features = [
    'PTS_last_3', 'EFF_last_3', 'USG_last_3',
    'PTS_season_avg', 'REB_season_avg', 'EFF_season_avg',
    'PTS_vs_OPP_hist', 'EFF_vs_OPP_hist',
    'OPP_PTS_ALLOWED_PER_PLAYER', 'OPP_REB_ALLOWED_PER_PLAYER', 
    'SCORING_VACUUM',
    'AST_last_3', 'BLK_last_3', 'STL_last_3', 'REB_last_3',
    'AST_season_avg', 'BLK_season_avg', 'STL_season_avg',
    'TS_PCT_last_3', 'TS_PCT_season_avg',
    'MONTH',
    'TEAM_USG_RANK' 
]

# 3. CONNECT TO DATABASE & CALCULATE TEAM CONTEXT
conn = sqlite3.connect('../data/basketball.db')

# A. Find the team name using the first player in your lineup
team_query = "SELECT TEAM FROM ml_features WHERE PLAYER = ? ORDER BY DATE DESC LIMIT 1"
target_team = pd.read_sql(team_query, conn, params=(active_lineup[0],)).iloc[0]['TEAM']

# B. Calculate the team's historical average PPG
team_avg_query = "SELECT AVG(team_pts) as team_avg FROM (SELECT SUM(PTS) as team_pts FROM ml_features WHERE TEAM = ? GROUP BY DATE)"
team_avg = pd.read_sql(team_avg_query, conn, params=(target_team,)).iloc[0]['team_avg']

# C. Calculate Lineup Power, Vacuum, AND Dynamic Hierarchy!
placeholders = ",".join(["?"] * len(active_lineup))
lineup_query = f"""
    SELECT PLAYER, PTS_season_avg, USG_season_avg 
    FROM ml_features 
    WHERE PLAYER IN ({placeholders}) 
    GROUP BY PLAYER HAVING MAX(DATE)
"""
lineup_df = pd.read_sql(lineup_query, conn, params=active_lineup)

# 1. Math for the Vacuum
lineup_power = lineup_df['PTS_season_avg'].sum()
calculated_vacuum = team_avg - lineup_power

# 2. Math for the Alpha Dog Rank (1.0 = Highest Usage on the court today)
lineup_df['CURRENT_USG_RANK'] = lineup_df['USG_season_avg'].rank(ascending=False, method='min')

# Create a dictionary to easily look up each player's rank later
# Looks like: {'Akis Kolovelonis': 1.0, 'Player Two': 2.0}
rank_dict = dict(zip(lineup_df['PLAYER'], lineup_df['CURRENT_USG_RANK']))


# D. Fetch the Opponent's Defense once for the whole team
o_query = "SELECT OPP_PTS_ALLOWED_PER_PLAYER, OPP_REB_ALLOWED_PER_PLAYER FROM ml_features WHERE OPPONENT = ? ORDER BY DATE DESC LIMIT 1"
o_df = pd.read_sql(o_query, conn, params=(upcoming_opponent,))

if o_df.empty:
    print(f"⚠️ Warning: Opponent '{upcoming_opponent}' not found. Using league averages.")
    opp_pts_allowed = 10.0 # Fallback safety
    opp_reb_allowed = 3.0  # Fallback safety
else:
    opp_pts_allowed = o_df['OPP_PTS_ALLOWED_PER_PLAYER'].iloc[0]
    opp_reb_allowed = o_df['OPP_REB_ALLOWED_PER_PLAYER'].iloc[0]

# 4. LOOP THROUGH LINEUP AND PREDICT
predictions_list = []
team_total = 0

for player in active_lineup:
    # Fetch player's latest baseline stats
    p_query = "SELECT * FROM ml_features WHERE PLAYER = ? ORDER BY DATE DESC LIMIT 1"
    p_df = pd.read_sql(p_query, conn, params=(player,))
    
    # Fetch Head-to-Head history
    h_query = "SELECT PTS_vs_OPP_hist, EFF_vs_OPP_hist FROM ml_features WHERE PLAYER = ? AND OPPONENT = ? ORDER BY DATE DESC LIMIT 1"
    h_df = pd.read_sql(h_query, conn, params=(player, upcoming_opponent))
    
    # Safety check if a new player is entered
    if p_df.empty:
        predictions_list.append((player, "No Data Found"))
        continue
        
    input_data = p_df.iloc[0].to_dict()
    
    # Inject Universal Game Context
    input_data['OPP_PTS_ALLOWED_PER_PLAYER'] = opp_pts_allowed
    input_data['OPP_REB_ALLOWED_PER_PLAYER'] = opp_reb_allowed
    input_data['SCORING_VACUUM'] = calculated_vacuum
    input_data['MONTH'] = month_of_game
    
    # NEW: Inject the dynamically calculated Usage Rank!
    # (If for some reason they aren't ranked, safely assume they are the 5th option)
    input_data['TEAM_USG_RANK'] = rank_dict.get(player, 5.0)
    
    # Inject Head-to-Head (Use NaN if they have never played this team)
    if not h_df.empty and pd.notna(h_df['PTS_vs_OPP_hist'].iloc[0]):
        input_data['PTS_vs_OPP_hist'] = h_df['PTS_vs_OPP_hist'].iloc[0]
        input_data['EFF_vs_OPP_hist'] = h_df['EFF_vs_OPP_hist'].iloc[0]
    else:
        input_data['PTS_vs_OPP_hist'] = np.nan
        input_data['EFF_vs_OPP_hist'] = np.nan

    # Format and Predict
    input_final = pd.DataFrame([input_data])[features].apply(pd.to_numeric, errors='coerce')
    pred = model.predict(input_final)[0]
    
    predictions_list.append((player, pred))
    team_total += pred

conn.close()

# ==========================================
# 5. PRINT THE FINAL BOX SCORE
# ==========================================
print("="*50)
print(f"🏀 TEAM PREDICTIONS: {target_team} vs {upcoming_opponent}")
print(f"Game Context: Month {month_of_game} | Vacuum: {calculated_vacuum:+.1f} pts")
print("="*50)

# Sort from highest projected scorer to lowest
valid_preds = [x for x in predictions_list if isinstance(x[1], (float, np.floating))]
missing_data = [x for x in predictions_list if not isinstance(x[1], (float, np.floating))]

# I PUT THIS LINE BACK IN so your box score sorts highest to lowest!

for player, proj in valid_preds:
    # Set to .1f for one decimal place (like 18.5)
    print(f"{player: <30} | {proj:>5.1f} pts")

for player, status in missing_data:
    print(f"{player: <30} | {status}")

print("-" * 50)
print(f"TOTAL PROJECTED SCORE:         | {team_total:>5.1f} pts")
print("="*50)

🏀 TEAM PREDICTIONS: Vouliagmeni Heat vs Hoston Celtics
Game Context: Month 2 | Vacuum: -7.2 pts
Konstantinos Spanakis          |   9.9 pts
Giorgos Lamprinidis            |   7.3 pts
Michail Alfaras                |  12.6 pts
Thodoris Lamprinidis           |   5.2 pts
Giannis Lamprinidis            |   8.6 pts
Nikolaos Lekakis               |   3.1 pts
Dionusis Miliopoulos           |   2.1 pts
--------------------------------------------------
TOTAL PROJECTED SCORE:         |  48.9 pts
